# Multi-Regional AeroMAPS - All Regions (Simple)

This notebook demonstrates running multiple regions sequentially using `MultiRegionalProcess`.

## Configuration

- **Mode**: `separate_processes` (sequential execution)
- **Regions**: 20 world regions (excluding France and Germany detailed configs)
- **Settings**: Default AeroMAPS parameters for all regions

## 0. Prepare Region Data

Generate partitioning files and config.yaml for each region from their CSV data.

In [ ]:
from pathlib import Path
from aeromaps.utils.functions import create_partitioning
import shutil

# Reference files from region_default
default_folder = Path("data/region_default")
energy_carriers_file = default_folder / "energy_carriers_data.yaml"

# All region folders (with actual folder names including typos and asterisks)
region_folders = [
    "region_af_dom", "region_af_int", "region_as_dom", "region_bras_dom",
    "region_eu_dom*", "region_eu_int*", "region_oc_dom", "region_oc_int",
    "region_other_as_int", "region_other_eur_dom", "region_other_eur_int",
    "region_other_nam_dom", "region_other_nam_int", "region_other_sam_dom",
    "region_sin_int", "region_uk_dom*", "region_uk_int*",
    "region_usa_dom", "region_usa_int", "region_sam_int"
]

data_path = Path("data")

for folder_name in region_folders:
    folder_path = data_path / folder_name
    
    if not folder_path.exists():
        print(f"⚠ Folder not found: {folder_name}")
        continue
    
    csv_file = folder_path / "dataframe_aeromaps.csv"
    if not csv_file.exists():
        print(f"⚠ CSV not found: {csv_file}")
        continue
    
    # 1. Generate partitioning from CSV (creates partitioning_updated_inputs.json)
    create_partitioning(file=str(csv_file), path=str(folder_path))
    
    # 2. Copy energy_carriers_data.yaml if not present
    dest_energy = folder_path / "energy_carriers_data.yaml"
    if not dest_energy.exists():
        shutil.copy(energy_carriers_file, dest_energy)
    
    # 3. Create config.yaml if not present (same format as region_france)
    config_file = folder_path / "config.yaml"
    if not config_file.exists():
        config_content = '''data:
  inputs:
    partitioning_data_file: "./partitioning_updated_inputs.json"
  outputs:
    json_outputs_file: "./outputs.json"

models:
  energy:
    energy_carriers_model_data_file: "./energy_carriers_data.yaml"
    resources_model_data_file: default
    processes_model_data_file: default

  standards:
    - models_traffic
    - models_efficiency_bottom_up
    - models_energy_without_fuel_effect
    - models_energy_cost
    - models_offset
    - models_emissions
'''
        config_file.write_text(config_content)
        
    print(f"✓ Prepared: {folder_name}")

print(f"\n✓ All {len(region_folders)} regions prepared")

## 1. Create Multi-Regional Process

In [ ]:
from aeromaps import create_multi_regional_process

# Create the multi-regional process
process = create_multi_regional_process(
    configuration_file="regionalisation_all_regions.yaml", disable_execution_statistics=True
)

print(f"✓ Process created")
print(f"  Mode: {process.execution_mode}")
print(f"  Regions ({len(process.list_regions())}): {process.list_regions()}")

## 2. Execute Sequential Computation

Run all regions sequentially with progress bar.

In [ ]:
import time

# Sequential execution (default)
start_time = time.time()
process.compute(parallel=False)  # Sequential mode
elapsed = time.time() - start_time

print(f"\n✓ Computation complete in {elapsed:.2f} seconds")

## 3. View Outputs

All outputs are stored with namespace prefixes in `data["vector_outputs"]`.

In [ ]:
# Global aggregated outputs
global_outputs = process.get_global_outputs()
print(f"Global outputs shape: {global_outputs.shape}")
global_outputs.head()

In [ ]:
# All outputs with namespaces
print(f"Total columns in vector_outputs: {len(process.data['vector_outputs'].columns)}")
process.data['vector_outputs'].columns.tolist()[:20]  # First 20

In [ ]:
# Regional outputs for a specific region
eu_dom = process.get_regional_outputs("EU_DOM")
print(f"EU_DOM outputs shape: {eu_dom.shape}")
eu_dom.head()

## 4. Emissions Analysis

Compare CO2 emissions metrics across all regions.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# Color palette - using a nice qualitative palette
colors = plt.cm.Set2.colors

# Plot 1: Global emissions comparison (top left)
ax1 = axes[0, 0]
years = process.data["vector_outputs"].index

# Use a gradient color palette for the waterfall
waterfall_colors = plt.cm.viridis([0.0, 0.2, 0.4, 0.6, 0.8])

# 1. CO2 emissions 2019 technology (BAU - no measures)
global_bau = "overall:co2_emissions_2019technology"
if global_bau in process.data["vector_outputs"].columns:
    process.data["vector_outputs"][global_bau].plot(
        ax=ax1, label="BAU (2019 tech)", linewidth=2.5, color=waterfall_colors[0]
    )

# 2. CO2 emissions including aircraft efficiency
global_eff = "overall:co2_emissions_including_aircraft_efficiency"
if global_eff in process.data["vector_outputs"].columns:
    process.data["vector_outputs"][global_eff].plot(
        ax=ax1, label="+ Aircraft efficiency", linewidth=2.5, color=waterfall_colors[1]
    )

# 3. CO2 emissions including load factor
global_lf = "overall:co2_emissions_including_load_factor"
if global_lf in process.data["vector_outputs"].columns:
    process.data["vector_outputs"][global_lf].plot(
        ax=ax1, label="+ Load factor", linewidth=2.5, color=waterfall_colors[2]
    )

# 4. CO2 emissions including energy
global_energy = "overall:co2_emissions_including_energy"
if global_energy in process.data["vector_outputs"].columns:
    process.data["vector_outputs"][global_energy].plot(
        ax=ax1, label="+ Energy", linewidth=2.5, color=waterfall_colors[3]
    )

# 5. CO2 emissions including energy + offset
global_offset = "overall:carbon_offset"
if global_energy in process.data["vector_outputs"].columns and global_offset in process.data["vector_outputs"].columns:
    net_co2 = process.data["vector_outputs"][global_energy] - process.data["vector_outputs"][global_offset]
    net_co2.plot(
        ax=ax1, label="+ Offset", linewidth=2.5, color=waterfall_colors[4]
    )

ax1.set_xlabel("Year")
ax1.set_ylabel("CO2 Emissions (Mt)")
ax1.set_title("Global CO2 Emissions - Scenario Comparison")
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Plot 2: All regions with base 1 in 2019 (top right)
ax2 = axes[0, 1]

# Use a nice color palette for many regions
regions = process.list_regions()
n_regions = len(regions)
cmap = plt.cm.tab20
region_colors = [cmap(i / n_regions) for i in range(n_regions)]

for idx, region_id in enumerate(regions):
    col_co2 = f"{region_id}:co2_emissions_including_energy"
    col_offset = f"{region_id}:carbon_offset"
    
    if col_co2 in process.data["vector_outputs"].columns and col_offset in process.data["vector_outputs"].columns:
        net_co2 = process.data["vector_outputs"][col_co2] - process.data["vector_outputs"][col_offset]
        
        # Normalize to base 1 in 2019
        base_2019 = net_co2.loc[2019]
        if base_2019 != 0:
            normalized = net_co2 / base_2019
            normalized.plot(ax=ax2, label=region_id, alpha=0.8, color=region_colors[idx], linewidth=1.5)

# Add global total (normalized)
if global_energy in process.data["vector_outputs"].columns and global_offset in process.data["vector_outputs"].columns:
    global_net = process.data["vector_outputs"][global_energy] - process.data["vector_outputs"][global_offset]
    base_2019_global = global_net.loc[2019]
    if base_2019_global != 0:
        normalized_global = global_net / base_2019_global
        normalized_global.plot(
            ax=ax2, label="GLOBAL", linewidth=3, linestyle="--", color="black"
        )

ax2.set_xlabel("Year")
ax2.set_ylabel("Relative CO2 Emissions (base 1 = 2019)")
ax2.set_title("CO2 Emissions Evolution by Region (normalized)")
ax2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5)

# Plot 3: ETS perimeter - UK_DOM and EU_DOM remaining emissions (bottom left)
ax3 = axes[1, 0]
ets_regions = ["UK_DOM", "EU_DOM"]
ets_colors = [colors[3], colors[4]]

for idx, region_id in enumerate(ets_regions):
    col_co2 = f"{region_id}:co2_emissions_including_energy"
    col_offset = f"{region_id}:carbon_offset"
    
    if col_co2 in process.data["vector_outputs"].columns and col_offset in process.data["vector_outputs"].columns:
        net_co2 = (process.data["vector_outputs"][col_co2] - process.data["vector_outputs"][col_offset]) * 1e6
        net_co2.plot(ax=ax3, label=region_id, linewidth=2.5, color=ets_colors[idx])

# Total ETS
ets_total = None
for region_id in ets_regions:
    col_co2 = f"{region_id}:co2_emissions_including_energy"
    col_offset = f"{region_id}:carbon_offset"
    if col_co2 in process.data["vector_outputs"].columns and col_offset in process.data["vector_outputs"].columns:
        net = (process.data["vector_outputs"][col_co2] - process.data["vector_outputs"][col_offset]) * 1e6
        if ets_total is None:
            ets_total = net.copy()
        else:
            ets_total += net

if ets_total is not None:
    ets_total.plot(ax=ax3, label="ETS Total", linewidth=3, linestyle="--", color="black")

ax3.set_xlabel("Year")
ax3.set_ylabel("ETS Allowances surrendered")
ax3.set_title("Required Allowances (UK + EU+)")
ax3.legend(loc="upper right", fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Plot 4: Cumulative offsets for all INT markets (bottom right)
ax4 = axes[1, 1]

# Find all INT regions
int_regions = [r for r in regions if "INT" in r]
int_colors = [plt.cm.Paired(i / len(int_regions)) for i in range(len(int_regions))]

cumul_total_offsets = None
for idx, region_id in enumerate(int_regions):
    col_offset = f"{region_id}:carbon_offset"
    
    if col_offset in process.data["vector_outputs"].columns:
        # Cumulative sum over years (convert Mt to tonnes)
        cumul_offset = process.data["vector_outputs"][col_offset].cumsum() * 1e6
        cumul_offset.plot(ax=ax4, label=region_id, linewidth=1.5, alpha=0.8, color=int_colors[idx])
        
        if cumul_total_offsets is None:
            cumul_total_offsets = cumul_offset.copy()
        else:
            cumul_total_offsets += cumul_offset

# Total cumulative offsets
if cumul_total_offsets is not None:
    cumul_total_offsets.plot(ax=ax4, label="Total INT", linewidth=3, linestyle="--", color="black")

ax4.set_xlabel("Year")
ax4.set_xlim(2020,2050)
ax4.set_ylabel("CORSIA Credits bought")
ax4.set_title("Cumulative CORSIA Credits Required")
ax4.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
process.data["vector_outputs"]["UK_DOM:energy_consumption"]/(process.data["vector_outputs"]["UK_DOM:energy_consumption"]+process.data["vector_outputs"]["UK_INT:energy_consumption"])